In [ ]:
from langchain_community.vectorstores import FAISS
from langgraph.graph.message import add_messages
from langgraph.graph import START,END,StateGraph
from langchain_community.document_loaders import TextLoader
from langchain_openai import ChatOpenAI,OpenAIEmbeddings
from langchain_core.tools import Tool
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq

In [2]:
langchain_loaded_data=TextLoader("langchain.txt",encoding="utf-8")
langchain_docs=langchain_loaded_data.load()

langgraph_loaded_data=TextLoader("langgraph.txt",encoding="utf-8")
langgraph_docs=langgraph_loaded_data.load()

In [3]:
embeddings=OpenAIEmbeddings(model="text-embedding-3-small")
llm=ChatOpenAI(model="gpt-4o-mini")

In [4]:
langchain_text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=100)
langgraph_text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=100)

langchain_chunks=langchain_text_splitter.split_documents(langchain_docs)
langgraph_chunks=langgraph_text_splitter.split_documents(langgraph_docs)

In [5]:
langchain_vectorstore=FAISS.from_documents(langchain_chunks,embedding=embeddings)
langgraph_vectorstore=FAISS.from_documents(langgraph_chunks,embeddings)

langchain_retriever=langchain_vectorstore.as_retriever();
langgraph_retriever=langgraph_vectorstore.as_retriever();

In [6]:
from langchain_core.tools import create_retriever_tool

langchain_retriever_tool=create_retriever_tool(
    langchain_retriever,
    "retriever_for_langchain",
    "Searches information about langchain"
)

langgraph_retriever_tool=create_retriever_tool(
    langgraph_retriever,
    "retriever_for_langgraph",
    "searches any information about langgraph"
)

In [7]:
tools=[langchain_retriever_tool,langgraph_retriever_tool]

In [8]:
from typing_extensions import TypedDict
from typing import Annotated,List
from langchain_core.messages import AnyMessage


class AgentState(TypedDict):
    messages:Annotated[List[AnyMessage],add_messages]

In [9]:
llm_with_tools=llm.bind_tools(tools)

In [10]:
def agent(state):
    """
    Invokes the agent model to generate a response based on the current state. Given the question,it will
    decide which retriever to use as a retriever using retriever tool,or simply end.

    Args:
        state(messages):The current state

    Returns:
      dict:the updated state with agent response appended to messages


    """

    messages=state["messages"]
    response=llm_with_tools.invoke(messages)

    return {"messages":[response]}